# ch04 传感器性能关联分析

**目标**：探究多路传感器信号间的耦合关系，评估不同工况下的信号稳定性

**依赖**：ch01

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('.') / '..'))
from src.utils.data_loader import load_condition_data, get_common_columns
from src.utils.metrics import calc_correlation_matrix, calc_stability_score
from src.utils.output_manager import save_dataframe

import pandas as pd
import numpy as np

CHAPTER_ID = 'ch04'

## 1. 数据加载

In [ ]:
conditions = {
    'normal': load_condition_data('normal'),
    'lineardrive': load_condition_data('lineardrive'),
    'pressure': load_condition_data('pressure'),
}

common_cols = get_common_columns(list(conditions.values()))
analog_cols = [c for c in common_cols if c.startswith('MotorData') and 'Set' not in c]
print(f"共同模拟量列: {len(analog_cols)} 路")

## 2. 相关性计算

In [ ]:
corr_results = {}
for cond_name, df in conditions.items():
    numeric_df = df[analog_cols].select_dtypes(include=[np.number])
    corr_results[cond_name] = calc_correlation_matrix(numeric_df, method='pearson')

# 显示 normal 工况的相关性矩阵
corr_results['normal']

## 3. 工况差异分析

In [ ]:
diff_records = []
for i, col1 in enumerate(analog_cols):
    for col2 in analog_cols[i+1:]:
        for cond_name in conditions.keys():
            diff_records.append({
                'signal_pair': f"{col1} vs {col2}",
                'condition': cond_name,
                'correlation': corr_results[cond_name].loc[col1, col2],
            })

diff_df = pd.DataFrame(diff_records)
diff_df

## 4. 信号稳定性计算

In [ ]:
stability_records = []
for cond_name, df in conditions.items():
    for col in analog_cols:
        if col in df.columns:
            scores = calc_stability_score(df[col], window=100)
            scores['condition'] = cond_name
            scores['signal'] = col
            stability_records.append(scores)

stability_df = pd.DataFrame(stability_records)
stability_df

## 5. 保存产物

In [ ]:
for cond_name, corr in corr_results.items():
    save_dataframe(corr.reset_index(), f'correlation_matrix_{cond_name}.csv', CHAPTER_ID)

save_dataframe(diff_df, 'correlation_diff_by_condition.csv', CHAPTER_ID)
save_dataframe(stability_df, 'signal_stability_scores.csv', CHAPTER_ID)